# Phase 7.4 — Production Architecture & Edge-Case Checks

This notebook verifies the architectural patterns and edge-case contracts defined in `reports/PHASE_7_4_PRODUCTION_ARCHITECTURE_AND_TEST_PLAN.md`:
1. **Option B Silent-Gateway Retention:** Verifies active assets with 0 recent telemetry rows are assigned `score=0.0` and `worst_metric="no_telemetry"`.
2. **Zero-Variance & Insufficient History Handling:** Verifies sample std with `ddof=1` and `.replace(0, np.nan)` safely produces `score=0.0`.
3. **Deterministic Tie-Breaking:** Verifies sorting on `(score desc, canonical gateway_id asc)`.
4. **Submission Schema Conformance:** Validates 120-row formatting against `validate_submission.py`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import datetime as dt
import numpy as np
import pandas as pd
from pathlib import Path

from nexora.data_loader import DataLoader, normalize_gateway_id
from nexora.backtesting.strategies import Baseline3SigmaStrategy
import validate_submission

print('Environment and libraries initialized successfully.')

In [ ]:
# Architectural Check 1: Verify Option B Silent Gateway Behavior on real data
data_dir = Path('../data')
dl = DataLoader(data_dir)
master = dl.load_master()
strategy = Baseline3SigmaStrategy(data_dir)

# Test on 2026-02-09 where gateway 0EA061007895 was installed on Monday and was completely silent
monday = dt.date(2026, 2, 9)
t = dt.datetime.combine(monday, dt.time.min, tzinfo=dt.timezone.utc)
eligible = master[
    (master['installed_on_dt'] <= t) & 
    ((master['decommissioned_on_dt'] > t) | master['decommissioned_on_dt'].isna())
].copy()

ranked = strategy.rank(eligible, monday)
silent_row = ranked[ranked['gateway_id'] == '0EA061007895']
print('Silent gateway ranking row under Option B:')
print(silent_row)
assert len(silent_row) == 1
assert silent_row['score'].iloc[0] == 0.0
assert silent_row['rank'].iloc[0] > 260
print('Option B silent gateway behavior successfully verified!')

In [ ]:
# Architectural Check 2: Deterministic Tie-Breaking Invariant
test_df = pd.DataFrame({
    'gateway_id': ['0E1B6F4DBA34', '064AED1EC0AA', '06F49BD8F572'],
    'score': [26.0, 26.0, 26.0]
})
sorted_df = test_df.sort_values(by=['score', 'gateway_id'], ascending=[False, True]).reset_index(drop=True)
print('Deterministic tie-break result:')
print(sorted_df)
assert list(sorted_df['gateway_id']) == ['064AED1EC0AA', '06F49BD8F572', '0E1B6F4DBA34']
print('Deterministic sorting invariant confirmed!')

In [ ]:
# Architectural Check 3: Schema Contract Verification
REQUIRED_COLUMNS = ['week_start', 'rank', 'gateway_id', 'score', 'reason']
print('Grader required columns:', REQUIRED_COLUMNS)
assert validate_submission.REQUIRED_COLUMNS == REQUIRED_COLUMNS
assert validate_submission.VISITS_PER_WEEK == 15
assert len(validate_submission.SCORED_WEEKS) == 8
print('Submission contract constants successfully validated against validate_submission.py.')